In VS Code, select `playground/.venv/bin/python` as the notebook kernel, then run the cells normally.

In [1]:
from pathlib import Path
import sys

# VS Code may start the kernel either here or in the repository root.
repository_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'playground' / '__init__.py').is_file()),
    None,
)
if repository_root is None:
    raise RuntimeError('Open this notebook from inside the MSAlign repository')
sys.path.insert(0, str(repository_root))

# MSAlign demo on custom inputs


**Notes on expected inputs:**

Provide a centroided MS/MS peak list, sorted by increasing `m/z`. Centroiding, deisotoping, noise removal, duplicate-peak merging, salt removal, and parent-molecule selection are not performed here. Intensities need not be normalized. `precursor_mz` is the measured ion m/z; do not insert it into `peaks`, because the DreaMS transform adds its own precursor pseudo-peak. Candidate strings must be valid RDKit SMILES for neutral molecules.

Inside `MSAlign`, DreaMS keeps the 100 most intense peaks, normalizes their intensities, pads the peak list, and prepends the precursor. Fused MSAlign+ also creates 0.1-Th spectrum bins, radius-2 4096-bit Morgan fingerprints, candidate exact masses, and the calibrated score combination. MolDeBERTa and DreaMS embeddings are computed on demand. Missing metadata uses learned missing/unknown encodings; a missing or unsupported adduct makes the MSAlign+ mass channels zero. See [`playground/README.md`](README.md) for the complete input contract.

In [2]:
spectrum = {
    # Centroided [m/z, intensity] peaks. Intensities need not be normalized.
    'peaks': [
        [43.0184, 0.18],
        [77.0390, 0.12],
        [105.0335, 0.46],
        [121.0284, 1.00],
        [138.0550, 0.31],
    ],
    'precursor_mz': 181.0495,
    # Both fields below are optional and may also be None.
    'adduct': '[M+H]+',
    'collision_energy': 30.0,
}

candidate = 'CC(=O)Oc1ccccc1C(=O)O'
candidates = [candidate, 'CCO', 'CCCO', 'c1ccccc1']

In [ ]:
import os
os.environ["HF_TOKEN"] = ...

You can find the pretrained MSAlign weights at the [Zenodo record 22830464](https://zenodo.org/records/22830464)


In [4]:
from playground import MSAlign

# Replace this path with an MSAlign+ bundle to use score fusion.
checkpoint = repository_root / 'checkpoints' / 'msalign_massspecgym_formula_seed0'
model = MSAlign(checkpoint)

/home/krzakala/Documents/github/MSAlign-NeurIPS2026/playground/.venv/lib/python3.11/site-packages/lightning/fabric/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
/home/krzakala/Documents/github/MSAlign-NeurIPS2026/playground/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
score = model(spectrum, candidate)
score

13.38502025604248

In [6]:
# The returned scores follow the input SMILES order.
scores = model(spectrum, candidates)
list(zip(candidates, scores))

[('CC(=O)Oc1ccccc1C(=O)O', 13.385017395019531),
 ('CCO', 10.078739166259766),
 ('CCCO', 11.78055191040039),
 ('c1ccccc1', 13.128549575805664)]

## Shared-space embeddings

A raw MSAlign checkpoint can also encode batches of spectra and molecules as unit-normalized vectors. Their inner product is the cosine similarity; MSAlign divides it by its learned temperature $\tau$ to produce the score:

$$\operatorname{score}(s,m)=\frac{\langle \operatorname{encode\_ms}(s),\operatorname{encode\_mol}(m)\rangle}{\tau}. $$

In [17]:
ms_embedding = model.encode_ms([spectrum])
mol_embedding = model.encode_mol([candidate])

score_from_embeddings = ms_embedding[0].dot(mol_embedding[0]).item() / model.temperature

print(score_from_embeddings)
print(model(spectrum, candidate))

13.38502129304796
13.38502025604248


# Demo on MassSpecGym

To test MSAlign on MassSpecGym data you can directly get the spectra, candidates and splits from the 
[Zenodo record 22830464](https://zenodo.org/records/22830464)

You just need to download massspecgym.zip and unzip in data/massspecgym/

In [8]:
import numpy as np
import json
import pandas as pd

# Load the spectras
print("MassSpecGym spectras: ")
spectra = np.load("../data/massspecgym/spectra.npy")
print(spectra.shape)

# Load the candidate map (dictionnary mapping SMILES to candidate lists)
candidate_map = json.load(open("../data/massspecgym/candidates/official_candidates_by_mass/map.json"))
print("Candidate maps: ")
key = "COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1"
ex = candidate_map[key]
print(f"Example candidates for {key}: ")
print(ex)

# Load the metadata
unique_smiles_list = pd.read_csv("../data/massspecgym/unique_smiles.csv")['smiles'].tolist()
metadata = pd.read_csv("../data/massspecgym/metadata.csv")
# Retrieve the SMILES strings for each spectra from the list of unique SMILES strings 
metadata['smiles'] = metadata['unique_smiles_idx'].apply(lambda x: unique_smiles_list[x])
# Keep only the relevant columns in the metadata
metadata = metadata[['collision_energy','adduct', 'precursor_mz', 'smiles']]
print("Metadata: ")
print(metadata.head(5))

MassSpecGym spectras: 
(231104, 100, 2)
Candidate maps: 
Example candidates for COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1: 
['COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1', 'O=C(C=CC=Cc1ccc2c(c1)OCO2)N1CCOCC1', 'CCN(CCCN(C)C)c1ncnc(Cl)c1[N+](=O)[O-]', 'O=C(O)C1(Cc2ccc(C(F)(F)F)cc2)CCCNC1', 'CC1C(=O)CCC2(O)C(=O)NC(C(=O)c3ccccc3)C12', 'COc1c(N)cccc1C(=O)OCOCc1ccccc1', 'CC(=CC(=O)NCCCO)c1ccccc1C(F)(F)F', 'CCc1cccc([N+](=O)[O-])c1COc1ccc(O)cc1C', 'COc1ccc(CCOCc2ccc([N+](=O)[O-])cc2)cc1', 'CN(C)C=Nc1nc(ON=Cc2ccccc2)ncc1F', 'COC(=O)c1cc(N)ccc1Oc1ccc(CCO)cc1', 'Cc1cc2ccccc2nc1-c1ccc(-c2nn[nH]n2)cc1', 'Cc1cccc(NC(=O)C2C3C=CC(O3)C2C(=O)O)c1C', 'COc1ccc2c(c1)c(=O)cc(C(=O)O)n2C1CCCC1', 'COC(C[N+](=O)[O-])c1cccc(OCc2ccccc2)c1', 'Cc1oc2cc3oc(=O)ccc3c(COCCN)c2c1C', 'COc1ccc(C(=O)c2cccc(N)c2)c(OC)c1OC', 'CCOc1ccc(C(=O)Nc2ccccc2O)cc1OC', 'CCOC(=O)CC1c2cc(C)ccc2NC1C(F)(F)F', 'COc1cccc(OC)c1C(=O)Nc1ccc(O)c(C)c1', 'O=C(O)C1CC2CN(C(=O)c3ccccc3)CC(C1)C2=O', 'NC(Cc1ccc(O)cc1Cc1ccc(O)cc1)C(=O)O', 'COc1cc(-c2ccc(C=O)o2)ccc1

In [9]:
from playground import MSAlign

# Replace this path with an MSAlign+ bundle to use score fusion.
checkpoint = repository_root / 'checkpoints' / 'msalign_massspecgym_formula_seed0'
model = MSAlign(checkpoint)

In [10]:
spectra_idx = np.random.randint(len(spectra))

# Get an input
spectra_example = {"peaks": spectra[spectra_idx],
                   "collision_energy": metadata.iloc[spectra_idx]['collision_energy'],
                   "adduct": metadata.iloc[spectra_idx]['adduct'],
                   "precursor_mz": metadata.iloc[spectra_idx]['precursor_mz']}

# Get the candidates
smiles_example = metadata.iloc[spectra_idx]['smiles']
candidates_example = candidate_map[smiles_example]
target_position = candidates_example.index(smiles_example)

# Get MSAlign scores
scores = model(spectra_example, candidates_example)

# Target rank
target_rank = scores.index(scores[target_position])
print(f"Target rank for spectra index {spectra_idx}: {target_rank}/{len(scores)}")


Target rank for spectra index 71900: 0/256


In [11]:
spectra_example

{'peaks': array([[1.84073822e+02, 1.00000000e+00],
        [1.25000206e+02, 3.47906798e-02],
        [1.84100967e+02, 7.04763457e-03],
        [1.04107132e+02, 7.01768044e-03],
        [1.84040497e+02, 6.97458023e-03],
        [1.84033051e+02, 6.58235047e-03],
        [9.89841766e+01, 6.48014760e-03],
        [1.84108734e+02, 6.31351583e-03],
        [1.84116028e+02, 6.13272516e-03],
        [1.84123566e+02, 5.66017115e-03],
        [1.84025635e+02, 5.59846079e-03],
        [1.84047974e+02, 5.04871365e-03],
        [1.84018082e+02, 4.96175699e-03],
        [4.96342072e+02, 4.93061962e-03],
        [1.84131210e+02, 4.93011205e-03],
        [1.84138626e+02, 4.67215339e-03],
        [1.84010452e+02, 4.61210497e-03],
        [1.85077637e+02, 4.36742650e-03],
        [1.84002914e+02, 4.23071301e-03],
        [1.83995331e+02, 3.89999431e-03],
        [1.84153778e+02, 3.81297432e-03],
        [1.84146454e+02, 3.78722325e-03],
        [1.84161285e+02, 3.72434128e-03],
        [1.83972382e+02, 